# 14 · The cleaning plan — review it, measure it, change it, then clean

**Use case.** After schema detection Langsat writes a **cleaning plan**: one decision per column (keep / transform / drop, with the operations and the reason) that compiles to the SQL the clean step runs. Chapter 01 accepted it as generated. This chapter stops before the clean, reads the plan and its SQL, measures a candidate change on a sample, edits three columns by hand — a placeholder token to NULL, an HTML entity, a brand prefix stripped with a regex — saves the overlay and runs the clean. It also shows the one thing an API key cannot do, and why.

**What you will learn**
1. Stop after `schema.detect()` — the plan is `pending_review`, nothing has run yet
2. Read the generated plan: per-column actions, ops and reasons, the compiled SQL, corrections and warnings
3. `plan(measured=True)`: what each decision does to null rates and distinct values on a sample
4. Find what the LLM missed by looking at the data yourself (`\N`, `&amp;`, `Visit Amazon's … Page`)
5. `preview()` a candidate overlay, then `save()` it and `clean()` — the plan is yours, the SQL is compiled from it
6. Hand-written SQL (`raw_expr`) is refused for an API key by design — what the app's SQL editor is for
7. `reset()` back to the generated plan; the data-model workspace next door

**What this costs.** nothing — plan reads, previews and a clean under 500K rows are free.

> Every cell below ran for real against `api.langsat.ai` — the outputs are what the API returned. Re-running is safe:
> projects are found by name and reused, and a finished model is not retrained.

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show, fig, metrics_table, project_models
from langsat import viz
import pandas as pd

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST-SDK-4' with 20 scopes


## 1 · Stop before the clean

`get_or_create_project(..., clean=False)` runs upload and schema detection and stops. The project is `schema_done`; the tables are readable as a head sample only; the cleaning plan exists but has not been applied — the status the app shows as *Review cleaning*.

In [2]:
p = get_or_create_project(ls, "cleaning-plan", kind="data_analysis", clean=False, fresh=True)
print("status:", p.status, "· cleaning decided:", bool(p.data.get("cleaning_decided")))

deleted the previous amazon-reviews-cleaning-plan — starting from nothing


created project 0d64a986-48a4-4bef-86aa-c5bfb5fb4fb6 (amazon-reviews-cleaning-plan, data_analysis)
uploading ['customer.csv', 'product.csv', 'review.csv'] …


   customer.csv: 9,821 rows × 2 columns → table 'customer'
   product.csv: 8,607 rows × 6 columns → table 'product'
   review.csv: 10,000 rows × 7 columns → table 'review'
detecting schema (primary keys, foreign keys, time column) …


stopping before the clean (clean=False) — the cleaning plan is yours to review
project ready: status=schema_done · type=data_analysis · files=['customer.csv', 'product.csv', 'review.csv']
status: schema_done · cleaning decided: False


## 2 · Read the plan Langsat wrote

`cleaning.plan()` returns the plan (`{table: {columns: {col: {action, ops, reason, locked…}}}}`), any **corrections** the validator made to the LLM's proposal (a key column it refused to drop, an op that does not fit the type), **warnings**, and `compiled_sql` — the exact DuckDB SQL the clean will run, generated from the plan. Locked columns are keys the graph depends on: they can be transformed, not dropped.

In [3]:
pl = p.cleaning.plan()
print("status:", pl["status"], "· tables:", list(pl["plan"]["tables"]), "· corrections:", len(pl["corrections"]), "· warnings:", len(pl["warnings"]))
rows = []
for table, t in pl["plan"]["tables"].items():
    for col, c in t["columns"].items():
        rows.append({"table": table, "column": col, "type": c["source_type"], "action": c["action"], "locked": c["lock_reason"] or "",
                     "ops": " → ".join(o["op"] + (f"({o['params']})" if o.get("params") else "") for o in c["ops"]),
                     "reason": (c["ops"][0].get("reason") if c["ops"] else "")[:70]})
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

status: pending_review · tables: ['review', 'product', 'customer'] · corrections: 0 · warnings: 0


,table,column,type,action,locked,ops,reason
0,review,rating,BIGINT,keep,,,
1,review,summary,VARCHAR,keep,,,
2,review,verified,VARCHAR,transform,,"boolean_to_int({'true_values': ['True'], 'false_values': ['False']})",Convert True/False string values to 0/1 integers for consistent boolea
3,review,product_id,BIGINT,keep,join_key,,
4,review,customer_id,BIGINT,keep,join_key,,
5,review,review_text,VARCHAR,keep,,,
6,review,review_time,TIMESTAMP,keep,time_col,,
7,product,brand,VARCHAR,keep,,,
8,product,price,DOUBLE,keep,,,
9,product,title,VARCHAR,keep,,,


In [4]:
print(pl["compiled_sql"])

-- LANGSAT_CLEANING_FORMAT: v2-duckdb-sql

CREATE OR REPLACE TABLE "cleaned_customer" AS
SELECT
    /*@col:customer_id*/ "customer_id" AS "customer_id",
    /*@col:customer_name*/ "customer_name" AS "customer_name"
FROM "raw_customer";

CREATE OR REPLACE TABLE "cleaned_product" AS
SELECT
    /*@col:brand*/ "brand" AS "brand",
    /*@col:price*/ "price" AS "price",
    /*@col:title*/ "title" AS "title",
    /*@col:category*/ "category" AS "category",
    /*@col:product_id*/ "product_id" AS "product_id",
    /*@col:description*/ CASE WHEN LOWER(TRIM("description")) IN ('n/a', 'nan', 'null') THEN NULL ELSE "description" END AS "description"
FROM "raw_product";

CREATE OR REPLACE TABLE "cleaned_review" AS
SELECT
    /*@col:rating*/ "rating" AS "rating",
    /*@col:summary*/ "summary" AS "summary",
    /*@col:verified*/ CASE WHEN LOWER(TRIM("verified")) IN ('true') THEN 1 WHEN LOWER(TRIM("verified")) IN ('false') THEN 0 ELSE NULL END AS "verified",
    /*@col:product_id*/ "product_id" AS "p

## 3 · Measure it on a sample

`plan(measured=True)` runs the plan on a sample and reports, per column, the null rate before and after, how many distinct values collapse, and anything suspicious (a cast that would null most of a column, a token list that catches real values).

In [5]:
measured = p.cleaning.plan(measured=True)
prev = pd.DataFrame(measured["preview"])
print("sample rows:", measured["preview_sample_rows"])
cols = [c for c in ("table", "column", "null_rate_before", "null_rate_after", "distinct_before", "distinct_after", "note") if c in prev.columns]
prev[cols] if len(prev) else prev

sample rows: 1000


,table,column,null_rate_before,null_rate_after,distinct_before,distinct_after
0,review,rating,0.000,0.000,5,5
1,review,summary,0.000,0.000,837,837
2,review,verified,0.000,0.000,2,2
3,review,product_id,0.000,0.000,906,906
4,review,customer_id,0.000,0.000,1015,1015
5,review,review_text,0.000,0.000,943,943
6,review,review_time,0.000,0.000,724,724
7,product,brand,0.001,0.001,784,784
8,product,price,0.000,0.000,267,267
9,product,title,0.000,0.000,1353,1353


## 4 · Look at the data yourself

The LLM saw a sample and column names; you know the source. Until the clean has run, the project's tables are not readable (`table()` is the *cleaned* data), so look at the file you uploaded. Three things in `product` the plan did not touch: the category column carries the export's `\N` placeholder for "no category", categories are HTML-escaped (`Literature &amp; Fiction`), and the brand column is a page title (`Visit Amazon's Stephen King Page`) rather than a name.

In [6]:
from _common import DATA_DIR
product = pd.read_csv(DATA_DIR / "product.csv")               # the uploaded file, as it went in
print("category placeholders:", (product["category"] == "\\N").sum(), "· HTML-escaped:", product["category"].str.contains("&amp;", na=False).sum())
print("brand as a page title:", product["brand"].str.startswith("Visit Amazon's", na=False).sum(), "of", len(product))
product[["brand", "category"]].sample(5, random_state=3)

category placeholders: 348 · HTML-escaped: 5406
brand as a page title: 7553 of 8607


,brand,category
3267,Lee M. Hollander,Books|Literature &amp; Fiction|Poetry
8040,Visit Amazon's Sally Fallon Morell Page,"Books|Crafts, Hobbies &amp; Home|Home Improvement &amp; Design"
5075,Visit Amazon's Ally Carter Page,Books|Teen &amp; Young Adult|Literature &amp; Fiction
7429,Visit Amazon's Katie MacAlister Page,Books|Literature &amp; Fiction|United States
1729,Visit Amazon's Robert Hichens Page,Books|Literature &amp; Fiction|Action &amp; Adventure


## 5 · Change the plan — preview, then save

An **overlay** names only the columns you want to decide; everything else stays as generated. Each column gets an `action` and a list of ops from the platform's toolkit — `trim`, `case`, `replace`, `strip_chars`, `null_if_in`, `regexp_replace`, `cast`, `parse_number`, `impute`, `clip`, `round`, `date_trunc`, `split_part`, `substring`, `json_extract` … The same ops the app's per-column editor offers; the SQL is compiled from them, so nothing you type here runs unvalidated.

`preview()` measures the candidate without saving; `save()` stores it as the draft the next clean runs.

In [7]:
overlay = {"product": {"columns": {
    "category": {"action": "transform", "ops": [
        {"op": "null_if_in", "params": {"values": ["\\N", "N/A", ""]}, "reason": "the export's placeholder for no category"},
        {"op": "replace", "params": {"find": "&amp;", "replace": "&"}, "reason": "HTML-escaped ampersands"}]},
    "brand": {"action": "transform", "ops": [
        {"op": "regexp_replace", "params": {"pattern": "^Visit Amazon's (.*) Page$", "replacement": "\\1"}, "reason": "the author's name, not the page title"},
        {"op": "trim", "params": {}}]},
}}}
candidate = p.cleaning.preview(overlay)
cand = pd.DataFrame(candidate["preview"])
cand[[c for c in cand.columns if c in ("table", "column", "null_rate_before", "null_rate_after", "distinct_before", "distinct_after", "note")]].query("column in ['category', 'brand']") if len(cand) else candidate["warnings"]

,table,column,null_rate_before,null_rate_after,distinct_before,distinct_after
3,product,category,0.000,0.003,146,108
4,product,brand,0.001,0.001,784,1037


In [8]:
saved = p.cleaning.save(overlay)
print("status:", saved["status"], "· corrections:", saved["corrections"] or "none")
for col in ("category", "brand"):
    c = saved["plan"]["tables"]["product"]["columns"][col]
    print(f"  {col}: {c['action']} ← " + " → ".join(f"{o['op']} [{o.get('origin')}]" for o in c["ops"]))
print()
print("\n".join(l for l in saved["compiled_sql"].splitlines() if "@col:brand" in l or "@col:category" in l))

status: pending_review · corrections: none
  category: transform ← null_if_in [user] → replace [user]
  brand: transform ← regexp_replace [user] → trim [user]

    /*@col:brand*/ TRIM(REGEXP_REPLACE("brand", '^Visit Amazon''s (.*) Page$', '\1', 'g')) AS "brand",
    /*@col:category*/ REPLACE(CASE WHEN LOWER(TRIM("category")) IN ('', '\n', 'n/a') THEN NULL ELSE "category" END, '&amp;', '&') AS "category",


## 6 · Run the clean, check the result

`clean()` runs the saved plan — the same call chapter 01 made, now on a plan you reviewed. Then read the cleaned table back: the placeholders are NULL, the ampersands are real, the brand is a name.

In [9]:
p.cleaning.clean().wait(timeout=1800)
product2 = p.table("product").to_pandas()
print("status:", p.refresh().status)
print("category NULLs:", product2["category"].isna().sum(), "(was 0; placeholders:", (product["category"] == "\\N").sum(), ") · '&amp;' left:", product2["category"].str.contains("&amp;", na=False).sum())
print("brands still 'Visit Amazon's …':", product2["brand"].str.startswith("Visit Amazon's", na=False).sum())
pd.DataFrame({"before": product["brand"].head(5).values, "after": product2["brand"].head(5).values})

status: schema_done
category NULLs: 348 (was 0; placeholders: 348 ) · '&amp;' left: 0
brands still 'Visit Amazon's …': 0


,before,after
0,Donna Mabry,Donna Mabry
1,Visit Amazon's Luke Young Page,Luke Young
2,Emma Mills,Emma Mills
3,Visit Amazon's Estelle Ryan Page,Estelle Ryan
4,Visit Amazon's Jodi LaPalm Page,Jodi LaPalm


## 7 · What an API key may not do — and why

The plan above is **structured**: ops with parameters, compiled by the platform. The app's SQL editor also lets a signed-in person write a column expression or the whole table SQL by hand (`raw_expr`, `edited_sql`). That SQL runs on the machine that cleans and trains your data, so an API key — which can be a script, a CI job, a leaked file — is refused there, on purpose, until the box sandbox ships. The refusal is a typed error you can branch on.

In [10]:
from langsat import errors
try:
    p.cleaning.save({"product": {"columns": {"price": {"action": "transform", "ops": [{"op": "raw_expr", "params": {"sql": "ROUND(price, 0)"}}]}}}})
except errors.NeedsUserSession as e:
    print("NeedsUserSession ·", e.status, e.code, "·", str(e.message)[:160])
except errors.LangsatError as e:
    print(type(e).__name__, "·", e.status, e.code, "·", str(e.message)[:160])

NeedsUserSession · 403 needs_user_session · Editing the cleaning SQL directly is only available to a signed-in user, not to an API key.


## 8 · Back to the generated plan; the workspace next door

`reset()` discards the overlay and returns to Langsat's plan (the next clean runs that). The data-model workspace (`p.data_model`) is the other half of *Review*: edit primary / foreign keys and cell values as **pending** changes, then `refresh()` applies them together with the draft plan at the lane price — see the SDK reference.

In [11]:
print("data model:", {k: v for k, v in p.data_model.get().items() if k in ("has_pending", "pending_summary", "status")} or list(p.data_model.get().keys())[:8])
save_metrics(".", {"notebook": "14_cleaning_plan", "task": "cleaning plan · review, preview, edit, clean", "model": "—", "project_id": p.id,
                   "headline": {"columns_in_plan": len(rows), "llm_transforms": sum(1 for r in rows if r["action"] == "transform"),
                                "category_nulls_after": int(product2["category"].isna().sum()), "brand_titles_left": int(product2["brand"].str.startswith("Visit Amazon's", na=False).sum())}})

data model: ['enabled', 'schema_result', 'pending', 'tables', 'flags', 'cleaning', 'impact', 'refresh']
wrote results/metrics.json


PosixPath('results/metrics.json')

## Where to go next

- [01 · Set up and explore](../01_setup_and_explore/) — the clean accepted as generated
- [00 · Connect your data](../00_connect_your_data/) — `refresh()` re-runs the saved plan on new data
- SDK reference: `help(p.cleaning)`, `help(p.data_model)`